# 라벨 수정 사본 만들기

4명의 EDA에서 찾은 라벨 오류를 같은 폴더의 `jaeone_label_fixes.csv`에 정리했습니다. 이 노트북은 그 목록대로 고친 데이터 사본을 만듭니다.

- 원본(`data/raw`)은 읽기만 하고 고치지 않습니다.
    - README에 원본은 받은 그대로 두기로 되어 있습니다.
    - `data/` 폴더는 git이 추적하지 않아서, 원본을 고치면 되돌릴 수 없습니다.
- 사본은 `data/processed` 바로 아래에 원본과 같은 구조(`train_images`, `train_annotations`, `test_images`)로 만듭니다.
    - 이미지는 바꾸지 않고 그대로 복사합니다.
    - JSON은 모두 복사한 뒤, 수정 목록에 있는 annotation만 고칩니다.
- 노트북을 다시 실행하면 사본의 세 폴더를 지우고 원본에서 다시 만듭니다. 그래서 사본은 항상 지금의 수정 목록과 같은 상태가 됩니다. `data/processed` 안의 다른 파일은 건드리지 않습니다.

### 진행 순서

1. 경로 설정
2. 수정 목록 읽기
3. 원본 JSON 색인
4. 수정 목록과 원본 대조
5. 사본 만들기와 수정 적용
6. 결과 검사
7. 수정 전후 비교 그림
8. 클래스별 bbox 영역 모아 보기 (사본)

## 1. 경로 설정

- 노트북이 어느 폴더에 있어도 같은 경로를 쓰도록, `pyproject.toml`이 있는 폴더를 레포 루트로 잡습니다.
- 원본 폴더는 `data/raw` 아래에서 `train_images` 폴더가 있는 곳으로 정합니다. 압축을 푼 방식에 따라 `data/raw` 바로 아래일 수도 있고, 한 단계 아래 폴더일 수도 있기 때문입니다.
- 경로는 내 컴퓨터의 절대경로 대신 레포 루트 기준 상대경로로 출력합니다.

In [ ]:
import csv
import json
import shutil
from pathlib import Path

import pandas as pd

# 노트북 위치와 상관없이 같은 경로를 쓰도록, pyproject.toml이 있는 폴더를 레포 루트로 잡습니다.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists()), None)
assert ROOT is not None, "pyproject.toml이 있는 레포 폴더를 찾지 못했습니다. 레포 안에서 노트북을 실행하세요."

# 원본: data/raw 아래에서 train_images 폴더가 있는 곳
found = [p for p in sorted((ROOT / "data" / "raw").rglob("train_images")) if p.is_dir()]
assert len(found) == 1, f"data/raw 아래에 train_images 폴더가 1개여야 합니다. 찾은 수: {len(found)}"
RAW_DIR = found[0].parent

# 사본: data/processed 바로 아래에 원본과 같은 세 폴더
OUT_DIR = ROOT / "data" / "processed"
SUB_DIRS = ["train_images", "train_annotations", "test_images"]

# 수정 목록
FIX_CSV = ROOT / "notebooks" / "eda" / "label_fix" / "jaeone_label_fixes.csv"

for name in SUB_DIRS:
    assert (RAW_DIR / name).is_dir(), f"원본 폴더가 없습니다: {(RAW_DIR / name).relative_to(ROOT).as_posix()}"
assert FIX_CSV.exists(), f"수정 목록이 없습니다: {FIX_CSV.relative_to(ROOT).as_posix()}"


def rel(path):
    """레포 루트 기준 상대경로를 / 구분자로 돌려준다. (내 컴퓨터의 절대경로를 출력하지 않기 위해)"""
    return Path(path).relative_to(ROOT).as_posix()


print("원본 폴더 :", rel(RAW_DIR))
print("사본 폴더 :", rel(OUT_DIR))
print("수정 목록 :", rel(FIX_CSV))

## 2. 수정 목록 읽기

`jaeone_label_fixes.csv`는 고칠 annotation 하나가 한 줄입니다.

| 열 | 내용 |
|---|---|
| `no` | `eda_label_issues_by_member.csv`의 번호 |
| `file_name`, `category_id` | 고칠 annotation의 이미지 파일명과 클래스 |
| `action` | 고치는 방법 (아래 표) |
| `bbox_before`, `bbox_after` | 고치기 전과 후의 bbox `[x, y, w, h]` |
| `area_before`, `area_after` | 고치기 전과 후의 area |
| `problem`, `source` | 문제 설명과 근거 (노트북 섹션, 값의 출처) |

| `action` | 하는 일 |
|---|---|
| `add` | 빠진 알약의 box를 새 JSON으로 추가합니다. `bbox_before`, `area_before`는 빈칸입니다. |
| `bbox` | bbox를 `bbox_after`로 바꾸고, area도 `area_after`로 바꿉니다. |
| `area` | bbox는 그대로 두고 area만 `area_after`로 바꿉니다. |

- 이 파일은 엑셀에서도 한글이 깨지지 않도록 BOM이 있는 UTF-8로 저장했습니다. 그래서 `utf-8-sig`로 엽니다. `utf-8`로 열면 BOM 문자가 첫 열 이름 앞에 붙어 남아서 `no` 열을 찾지 못합니다.
- 읽은 뒤 아래를 확인하고, 하나라도 맞지 않으면 멈춥니다.
    - 열 이름이 위 표와 같은지
    - `action`이 `add`, `bbox`, `area` 중 하나인지
    - 같은 이미지의 같은 클래스가 두 줄 이상 없는지 (annotation 하나는 한 번만 고칩니다)
    - `area_after`가 `bbox_after`의 너비 × 높이와 같은지

In [ ]:
with open(FIX_CSV, encoding="utf-8-sig", newline="") as f:
    fix_df = pd.DataFrame(list(csv.DictReader(f)))

FIX_COLUMNS = ["no", "file_name", "category_id", "action", "bbox_before", "bbox_after",
               "area_before", "area_after", "problem", "source"]
assert list(fix_df.columns) == FIX_COLUMNS, f"열 이름이 다릅니다: {list(fix_df.columns)}"

# 글자로 읽힌 값을 숫자와 리스트로 바꿉니다. add 줄의 before 값처럼 빈칸인 곳은 비워 둡니다(bbox는 None, area는 <NA>).
fix_df["no"] = fix_df["no"].astype(int)
fix_df["category_id"] = fix_df["category_id"].astype(int)
for col in ["bbox_before", "bbox_after"]:
    fix_df[col] = fix_df[col].apply(lambda s: json.loads(s) if s else None)
for col in ["area_before", "area_after"]:
    # 정수 열에 빈칸이 섞이면 pandas가 실수(97965.0)로 바꾸므로, 빈칸을 허용하는 정수형(Int64)으로 둡니다.
    fix_df[col] = pd.array([int(s) if s else None for s in fix_df[col]], dtype="Int64")

assert set(fix_df["action"]) <= {"add", "bbox", "area"}, f"알 수 없는 action: {set(fix_df['action'])}"
assert not fix_df.duplicated(["file_name", "category_id"]).any(), "같은 이미지의 같은 클래스가 두 줄 이상 있습니다."
area_ok = fix_df["area_after"] == fix_df["bbox_after"].apply(lambda b: b[2] * b[3])
assert area_ok.all(), f"area_after가 bbox_after의 너비 × 높이와 다른 줄: {fix_df.loc[~area_ok, 'no'].tolist()}"

print(f"수정할 annotation {len(fix_df)}줄, 이미지 {fix_df['file_name'].nunique()}장")
display(fix_df["action"].value_counts().rename_axis("action").reset_index(name="줄 수"))

# 파일명이 잘리지 않도록 이 표에서만 열 너비 제한을 풉니다.
with pd.option_context("display.max_colwidth", None):
    display(fix_df[["no", "file_name", "category_id", "action", "bbox_before", "bbox_after",
                    "area_before", "area_after"]])

## 3. 원본 JSON 색인

수정 목록의 한 줄(이미지 파일명 + `category_id`)이 원본의 어느 JSON 파일인지 찾으려고, 원본 JSON을 모두 읽어 표(`raw_df`)로 만듭니다. 원본은 읽기만 합니다.

4번 이후 단계는 아래 내용이 맞다는 전제로 동작합니다. 여기서 모두 확인하고, 하나라도 맞지 않으면 멈춥니다.

| 확인 | 필요한 이유 |
|---|---|
| JSON마다 `images`, `annotations`, `categories`가 1개씩 있다 | JSON 하나를 알약 하나로 보고 고치기 때문입니다. |
| (이미지 파일명, `category_id`) 짝이 JSON마다 다르다 | 수정 목록의 한 줄이 JSON 하나만 가리켜야 합니다. |
| 약 폴더 이름(`K-` 코드), `annotations`의 `category_id`, `categories`의 `id`가 같다 | 14번은 두 JSON의 bbox만 맞바꿉니다. 그러면 폴더, 약 정보, 클래스가 그대로 맞는다는 전제입니다. |
| JSON 파일 이름과 이미지 파일 이름이 같다 (확장자만 다름) | 빠진 box의 새 JSON 파일 이름을 이미지 파일 이름으로 정하기 때문입니다. |
| annotation id가 서로 겹치지 않는다 | 새 annotation id를 원본에서 가장 큰 번호 다음부터 매기기 때문입니다. |
| train 이미지 파일과 JSON에 적힌 이미지가 같다 | 사본에서 짝이 맞지 않는 파일이 없어야 합니다. |

마지막으로 원본 JSON 하나의 앞부분을 그대로 출력해 저장 형식(들여쓰기, 한글 표기)을 봅니다. 5번에서 고친 JSON을 원본과 같은 형식으로 저장하려는 것입니다.

In [ ]:
ANN_DIR = RAW_DIR / "train_annotations"
IMG_DIR = RAW_DIR / "train_images"

records = []
for path in sorted(ANN_DIR.rglob("*.json")):
    with open(path, encoding="utf-8-sig") as f:
        data = json.load(f)
    counts = (len(data["images"]), len(data["annotations"]), len(data["categories"]))
    assert counts == (1, 1, 1), f"images, annotations, categories가 1개씩이 아닙니다 {counts}: {rel(path)}"
    img, ann, cat = data["images"][0], data["annotations"][0], data["categories"][0]
    records.append({
        "json_path": path.relative_to(ANN_DIR).as_posix(),  # train_annotations 폴더 기준 경로
        "folder_code": path.parent.name,                     # 약 폴더 이름 (K-코드)
        "file_name": img["file_name"],
        "image_id": img["id"],
        "category_id": ann["category_id"],
        "categories_id": cat["id"],
        "ann_id": ann["id"],
        "bbox": ann["bbox"],
        "area": ann["area"],
    })
raw_df = pd.DataFrame(records)

assert not raw_df.duplicated(["file_name", "category_id"]).any(), "같은 이미지의 같은 클래스 JSON이 2개 이상 있습니다."

folder_id = raw_df["folder_code"].str.removeprefix("K-").astype(int)
assert (folder_id == raw_df["category_id"]).all(), "약 폴더 이름과 category_id가 다른 JSON이 있습니다."
assert (raw_df["categories_id"] == raw_df["category_id"]).all(), "categories의 id와 category_id가 다른 JSON이 있습니다."

json_stem = raw_df["json_path"].str.rsplit("/", n=1).str[-1].str.removesuffix(".json")
assert (json_stem == raw_df["file_name"].str.removesuffix(".png")).all(), "JSON 파일 이름과 이미지 파일 이름이 다른 JSON이 있습니다."

assert raw_df["ann_id"].is_unique, "annotation id가 겹치는 JSON이 있습니다."

train_names = {p.name for p in IMG_DIR.glob("*.png")}
assert set(raw_df["file_name"]) == train_names, "train 이미지 파일과 JSON에 적힌 이미지가 다릅니다."

print(f"원본 JSON {len(raw_df)}개, 이미지 {raw_df['file_name'].nunique()}장, 클래스 {raw_df['category_id'].nunique()}개")
print(f"annotation id 범위: {raw_df['ann_id'].min()} ~ {raw_df['ann_id'].max()}")

# 저장 형식을 보려고 원본 JSON 하나의 앞부분을 그대로 출력합니다.
sample_path = ANN_DIR / raw_df.loc[0, "json_path"]
print()
print("원본 JSON 앞 8줄:", rel(sample_path))
print("\n".join(sample_path.read_text(encoding="utf-8-sig").splitlines()[:8]))

## 4. 수정 목록과 원본 대조

사본을 만들기 전에 수정 목록이 원본과 맞는지 확인합니다. 목록의 값이 원본과 다르면(줄을 잘못 적었거나, 원본이 이미 바뀌었거나) 엉뚱한 annotation을 덮어쓰게 되기 때문입니다. 하나라도 맞지 않으면 멈춥니다.

- `bbox`, `area` 줄: 원본 JSON의 bbox와 area가 목록의 `bbox_before`, `area_before`와 같은지 확인합니다.
- `add` 줄
    - 그 이미지에 해당 클래스의 JSON이 아직 없는지 확인합니다.
    - 이미지 파일명에 그 약 코드가 들어 있는지 확인합니다.
    - 새 JSON을 만들 때 값을 가져올 JSON이 있는지 확인합니다.
        - 약 정보: 같은 조합 폴더의 같은 약 폴더에 있는, 다른 각도 사진의 JSON
        - 이미지 정보: 같은 사진의 다른 알약 JSON
        - 해당하는 JSON이 여러 개면 파일 이름 순서로 첫 번째를 씁니다. 실행할 때마다 같은 JSON을 쓰기 위해서입니다.

아래 표 두 개를 출력합니다. 위 표는 `bbox`, `area` 줄의 대조 결과이고, 아래 표는 `add` 줄마다 값을 가져올 JSON입니다. 경로는 `train_annotations` 폴더 기준입니다.

In [ ]:
raw_lookup = {(r.file_name, r.category_id): r for r in raw_df.itertuples(index=False)}


def combo_of(file_name):
    """이미지 파일명에서 조합 이름(첫 번째 _ 앞부분)을 꺼낸다. 예: K-003351-013900-021325"""
    return file_name.split("_")[0]


change_checks = []  # bbox, area 줄의 대조 결과
add_sources = {}    # add 줄: (이미지 파일명, category_id) → (약 정보 JSON, 이미지 정보 JSON)
for r in fix_df.itertuples(index=False):
    key = (r.file_name, r.category_id)
    code = f"K-{r.category_id:06d}"

    if r.action == "add":
        assert key not in raw_lookup, f"{r.no}번: 원본에 이미 JSON이 있는데 add로 적혀 있습니다."
        assert f"{r.category_id:06d}" in combo_of(r.file_name).split("-")[1:], f"{r.no}번: 파일명에 {code}가 없습니다."

        # 약 정보: 같은 조합 폴더의 같은 약 폴더에 있는 다른 각도 사진의 JSON
        drug_dir = ANN_DIR / f"{combo_of(r.file_name)}_json" / code
        drug_jsons = sorted(p for p in drug_dir.glob("*.json") if p.stem != Path(r.file_name).stem)
        assert drug_jsons, f"{r.no}번: {code} 약 정보를 가져올 다른 각도 JSON이 없습니다."

        # 이미지 정보: 같은 사진의 다른 알약 JSON
        image_jsons = sorted(raw_df.loc[raw_df["file_name"] == r.file_name, "json_path"])
        assert image_jsons, f"{r.no}번: 이미지 정보를 가져올 같은 사진의 JSON이 없습니다."

        add_sources[key] = (drug_jsons[0].relative_to(ANN_DIR).as_posix(), image_jsons[0])
    else:
        assert key in raw_lookup, f"{r.no}번: 원본에 해당 JSON이 없습니다."
        raw = raw_lookup[key]
        change_checks.append({
            "no": r.no, "file_name": r.file_name, "category_id": r.category_id, "action": r.action,
            "원본 bbox": raw.bbox, "bbox_before와 같음": list(raw.bbox) == r.bbox_before,
            "원본 area": raw.area, "area_before와 같음": bool(raw.area == r.area_before),
        })

change_df = pd.DataFrame(change_checks)
add_df = pd.DataFrame([
    {"no": r.no, "file_name": r.file_name, "category_id": r.category_id,
     "약 정보 JSON": add_sources[(r.file_name, r.category_id)][0],
     "이미지 정보 JSON": add_sources[(r.file_name, r.category_id)][1]}
    for r in fix_df.itertuples(index=False) if r.action == "add"
])

with pd.option_context("display.max_colwidth", None):
    display(change_df)
    display(add_df)

bad = change_df.loc[~(change_df["bbox_before와 같음"] & change_df["area_before와 같음"]), "no"].tolist()
assert not bad, f"원본과 bbox_before 또는 area_before가 다른 줄: {bad}"
print(f"bbox, area 줄 {len(change_df)}개: 원본과 모두 같음")
print(f"add 줄 {len(add_df)}개: 새 JSON에 쓸 약 정보와 이미지 정보 JSON을 모두 찾음")

### 저장 형식 확인

3번 출력에서 원본 JSON이 4칸 들여쓰기로 저장된 것을 봤습니다. 다만 앞 8줄에는 한글이 없어서, 한글을 그대로 썼는지 `\uXXXX` 형태로 바꿔 썼는지는 알 수 없었습니다.

5번에서 고친 JSON을 원본과 같은 형식으로 저장하려고, 원본 JSON 763개를 모두 아래 기준으로 확인합니다.

- 파일 맨 앞에 BOM이 있는지
- 줄바꿈 문자가 CRLF(Windows 방식)인지 LF인지
- 파일 끝에 줄바꿈이 있는지
- 파이썬 `json.dumps(indent=4)`로 다시 썼을 때 원본과 글자 하나까지 같은지
    - `ensure_ascii=False`: 한글을 그대로 씀
    - `ensure_ascii=True`: 한글을 `\uXXXX` 형태로 씀

모든 JSON이 한 가지 형식이면 5번에서 그 형식으로 씁니다. 그러면 고친 JSON은 원본과 고친 값만 다르고, 나머지 글자는 같습니다.

In [ ]:
from collections import Counter

format_counts = Counter()
for json_path in raw_df["json_path"]:
    raw_bytes = (ANN_DIR / json_path).read_bytes()
    has_bom = raw_bytes.startswith(b"\xef\xbb\xbf")
    text = raw_bytes.decode("utf-8-sig")
    uses_crlf = "\r\n" in text
    body = text.replace("\r\n", "\n")
    ends_with_newline = body.endswith("\n")
    body = body.rstrip("\n")
    data = json.loads(body)
    # json.dumps(indent=4)로 다시 썼을 때 원본과 같아지는 ensure_ascii 값 (둘 다 아니면 None)
    same_with = next((ea for ea in (False, True) if json.dumps(data, ensure_ascii=ea, indent=4) == body), None)
    format_counts[(has_bom, uses_crlf, ends_with_newline, same_with)] += 1

format_df = pd.DataFrame(
    [{"BOM": bom, "줄바꿈": "CRLF" if crlf else "LF", "파일 끝 줄바꿈": end,
      "json.dumps(indent=4)로 재현": {False: "ensure_ascii=False (한글 그대로)",
                                     True: "ensure_ascii=True (\\uXXXX)", None: "재현 안 됨"}[ea],
      "JSON 수": n}
     for (bom, crlf, end, ea), n in format_counts.most_common()]
)
display(format_df)

## 5. 사본 만들기와 수정 적용

세 단계로 나눠 진행합니다.

| 단계 | 하는 일 |
|---|---|
| 5-1 | 빠진 box의 새 JSON에서, `images` 항목 중 어느 것을 같은 사진의 JSON에서 가져오고 어느 것을 같은 약의 JSON에서 가져올지 나누고 확인합니다. |
| 5-2 | 이전에 만든 사본 폴더 세 개를 지우고, 원본을 그대로 복사합니다. |
| 5-3 | 사본의 JSON에 수정 목록을 적용합니다. |

고친 JSON은 원본과 같은 형식으로 저장합니다. 4번의 저장 형식 확인에서 원본 763개가 모두 같은 형식이었습니다.

- BOM이 없고, 줄바꿈은 LF이며, 파일 끝에 줄바꿈이 없습니다.
- `json.dumps(indent=4, ensure_ascii=False)`로 다시 쓰면 원본과 글자 하나까지 같습니다.
- Windows에서 텍스트 모드로 저장하면 줄바꿈이 CRLF로 바뀝니다. 그래서 글자를 바이트로 바꿔 저장합니다.

### 5-1. 새 JSON에 쓸 항목 나누기

빠진 box 8개의 새 JSON은 `images` 항목 50개를 두 곳에서 가져옵니다.

- 이미지 정보 12개(`id`, `file_name`, `imgfile`, `width`, `height`, `size`, `back_color`, `light_color`, `drug_S`, `drug_dir`, `camera_la`, `camera_lo`): 같은 사진의 다른 알약 JSON에서 가져옵니다.
- 나머지 항목(약 정보): 같은 약의 다른 각도 JSON에서 가져옵니다.

이렇게 나누려면 아래 두 가지가 맞아야 합니다. 원본 JSON 763개로 확인하고, 맞지 않는 항목이 있으면 멈춥니다.

- 이미지 정보 12개는 같은 사진의 JSON끼리 값이 같아야 합니다. 사진에 대한 값이기 때문입니다.
- 나머지 항목은 같은 약(`category_id`)의 JSON끼리 값이 같아야 합니다. 그래야 다른 각도 JSON에서 가져온 값이 이 사진에서도 맞습니다.

`annotations`와 `categories`는 다음과 같이 만듭니다.

- `annotations`: 약 정보 JSON의 annotation을 바탕으로 4개 값만 바꿉니다.
    - `id`: 원본에서 가장 큰 번호(5691) 다음부터 매깁니다.
    - `image_id`: 이미지 정보 JSON의 `images`의 `id`
    - `bbox`, `area`: 수정 목록의 값
- `categories`와 `type`: 약 정보 JSON의 값을 그대로 씁니다. 같은 약이라 값이 같습니다.

In [ ]:
IMAGE_FIELDS = ["id", "file_name", "imgfile", "width", "height", "size",
                "back_color", "light_color", "drug_S", "drug_dir", "camera_la", "camera_lo"]

# 원본 JSON의 images 항목을 모두 읽습니다. 값 비교를 위해 값을 JSON 문자열로 바꿔 둡니다.
image_rows = []
for r in raw_df.itertuples(index=False):
    img = json.loads((ANN_DIR / r.json_path).read_text(encoding="utf-8"))["images"][0]
    row = {"_image": r.file_name, "_class": r.category_id}
    row.update({key: json.dumps(value, ensure_ascii=False) for key, value in img.items()})
    image_rows.append(row)
image_fields_df = pd.DataFrame(image_rows)

all_fields = [c for c in image_fields_df.columns if not c.startswith("_")]
assert image_fields_df[all_fields].notna().all().all(), "images 항목 구성이 다른 JSON이 있습니다."
assert set(IMAGE_FIELDS) <= set(all_fields), f"없는 이미지 정보 항목: {set(IMAGE_FIELDS) - set(all_fields)}"
drug_fields = [f for f in all_fields if f not in IMAGE_FIELDS]

# 같은 사진의 JSON끼리 값이 다른 이미지 정보 항목 / 같은 약의 JSON끼리 값이 다른 약 정보 항목
image_varying = [f for f in IMAGE_FIELDS if (image_fields_df.groupby("_image")[f].nunique() > 1).any()]
drug_varying = [f for f in drug_fields if (image_fields_df.groupby("_class")[f].nunique() > 1).any()]

print(f"images 항목 {len(all_fields)}개 = 이미지 정보 {len(IMAGE_FIELDS)}개 + 약 정보 {len(drug_fields)}개")
print("같은 사진의 JSON끼리 값이 다른 이미지 정보 항목:", image_varying or "없음")
print("같은 약의 JSON끼리 값이 다른 약 정보 항목:", drug_varying or "없음")
assert not image_varying and not drug_varying, "항목을 나누는 기준이 맞지 않습니다. 위 출력을 확인하세요."

### 5-2. 원본 복사

- `data/processed` 안의 `train_images`, `train_annotations`, `test_images` 세 폴더만 지우고, 원본에서 다시 복사합니다. `data/processed`의 다른 파일(`.gitkeep` 등)은 그대로 둡니다.
- 지우기 전에, 사본 폴더와 원본 폴더가 서로 겹치지 않는지 확인합니다. 겹치면 지우는 작업이 원본에 닿을 수 있기 때문입니다.
- 6번에서 원본이 바뀌지 않았는지 비교하려고, 복사하기 전에 원본 파일 전체의 MD5 해시를 계산해 둡니다. 이미지가 약 1.8GB라서 해시 계산과 복사에 시간이 조금 걸립니다.

In [ ]:
import hashlib


def file_hashes(folder):
    """폴더 안 모든 파일의 MD5 해시를 {폴더 기준 상대경로: 해시}로 돌려준다."""
    return {p.relative_to(folder).as_posix(): hashlib.md5(p.read_bytes()).hexdigest()
            for p in sorted(folder.rglob("*")) if p.is_file()}


# 6번에서 원본이 바뀌지 않았는지 비교하려고, 복사 전 원본의 해시를 저장해 둡니다.
raw_hashes_before = {name: file_hashes(RAW_DIR / name) for name in SUB_DIRS}

# 사본 폴더와 원본 폴더가 같거나, 한쪽이 다른 쪽 안에 있으면 멈춥니다.
out_real, raw_real = OUT_DIR.resolve(), RAW_DIR.resolve()
assert out_real != raw_real and out_real not in raw_real.parents and raw_real not in out_real.parents, \
    "사본 폴더와 원본 폴더가 겹칩니다."

for name in SUB_DIRS:
    target = OUT_DIR / name
    if target.exists():
        shutil.rmtree(target)  # 이전 실행에서 만든 사본 폴더만 지웁니다.
    shutil.copytree(RAW_DIR / name, target)
    n_copied = sum(1 for p in target.rglob("*") if p.is_file())
    print(f"{rel(target)}: 파일 {n_copied}개 (원본 {len(raw_hashes_before[name])}개)")

### 5-3. 수정 적용

수정 목록의 줄마다 `action`에 따라 사본의 JSON을 고칩니다. 원본 JSON은 열어서 읽기만 합니다.

| `action` | 하는 일 |
|---|---|
| `bbox` | 사본 JSON의 bbox와 area를 `bbox_after`, `area_after`로 바꿉니다. |
| `area` | 사본 JSON의 area만 `area_after`로 바꿉니다. |
| `add` | 5-1의 방식으로 새 JSON을 만들어, 원본과 같은 폴더 구조(`<조합>_json/K-<약 코드>/<이미지 이름>.json`)로 저장합니다. |

결과 표에는 줄마다 사본 JSON의 경로(`train_annotations` 기준), 적용한 bbox와 area, annotation id를 보여 줍니다.

In [ ]:
OUT_ANN_DIR = OUT_DIR / "train_annotations"


def read_json(path):
    return json.loads(path.read_text(encoding="utf-8"))


def write_json(path, data):
    """원본과 같은 형식(4칸 들여쓰기, 한글 그대로, BOM 없음, LF, 파일 끝 줄바꿈 없음)으로 저장한다."""
    path.write_bytes(json.dumps(data, ensure_ascii=False, indent=4).encode("utf-8"))


next_ann_id = int(raw_df["ann_id"].max()) + 1
applied = []
for r in fix_df.itertuples(index=False):
    key = (r.file_name, r.category_id)

    if r.action in ("bbox", "area"):
        json_rel = raw_lookup[key].json_path
        path = OUT_ANN_DIR / json_rel
        data = read_json(path)
        ann = data["annotations"][0]
        if r.action == "bbox":
            ann["bbox"] = list(r.bbox_after)
        ann["area"] = int(r.area_after)
        write_json(path, data)

    else:  # add
        drug_rel, image_rel = add_sources[key]
        data = read_json(ANN_DIR / drug_rel)                   # 약 정보: 같은 약의 다른 각도 JSON (원본)
        image_info = read_json(ANN_DIR / image_rel)["images"][0]  # 이미지 정보: 같은 사진의 다른 알약 JSON (원본)
        for field in IMAGE_FIELDS:
            data["images"][0][field] = image_info[field]
        ann = data["annotations"][0]
        assert ann["category_id"] == r.category_id, f"{r.no}번: 약 정보 JSON의 category_id가 다릅니다."
        ann["id"] = next_ann_id
        ann["image_id"] = image_info["id"]
        ann["bbox"] = list(r.bbox_after)
        ann["area"] = int(r.area_after)
        next_ann_id += 1

        json_rel = f"{combo_of(r.file_name)}_json/K-{r.category_id:06d}/{Path(r.file_name).stem}.json"
        path = OUT_ANN_DIR / json_rel
        assert not path.exists(), f"{r.no}번: 새로 만들 JSON이 이미 있습니다."
        write_json(path, data)

    applied.append({"no": r.no, "action": r.action, "사본 JSON": json_rel,
                    "bbox": ann["bbox"], "area": ann["area"], "annotation id": ann["id"]})

applied_df = pd.DataFrame(applied)
with pd.option_context("display.max_colwidth", None):
    display(applied_df)

n_json = sum(1 for _ in OUT_ANN_DIR.rglob("*.json"))
print(f"고친 JSON {(applied_df['action'] != 'add').sum()}개, 새로 만든 JSON {(applied_df['action'] == 'add').sum()}개")
print(f"사본 JSON 수: {n_json}개 (원본 {len(raw_df)}개)")

## 6. 결과 검사

사본이 의도대로 만들어졌는지 두 단계로 확인합니다. 검사마다 기대값과 결과를 표로 보여 주고, 하나라도 다르면 멈춥니다.

### 6-1. 사본 라벨 검사

사본 JSON을 모두 다시 읽어서, EDA에서 찾았던 문제 유형이 사본에 남아 있지 않은지 셉니다.

| 검사 | 기대값 | 이유 |
|---|---|---|
| 사본 JSON 수 | 771 | 원본 763개 + 새로 만든 8개 |
| 수정 목록대로 바뀌지 않은 줄 | 0 | 21줄이 모두 사본에 반영됐는지 확인합니다. |
| 파일명의 약 코드와 annotation의 클래스가 다른 이미지 | 0 | 원본에서는 라벨 누락 8장이 여기에 해당했습니다. |
| 이미지 밖 bbox | 0 | 원본에는 1개 있었습니다(18357, x = 6567). |
| area가 너비 × 높이와 다른 annotation | 0 | 원본에는 6개 있었습니다. |
| 같은 이미지에서 좌표가 같은 서로 다른 클래스의 bbox 쌍 | 0 | 원본에는 3쌍 있었습니다. |
| annotation id 중복 | 0 | 새 annotation id가 기존 번호와 겹치지 않는지 확인합니다. |
| image_id가 2개 이상인 이미지 | 0 | 새 JSON의 image_id가 같은 사진의 다른 JSON과 같은지 확인합니다. |
| 약 폴더 이름, category_id, categories의 id가 다른 JSON | 0 | 14번 bbox 맞바꾸기와 새 JSON의 저장 위치가 맞는지 확인합니다. |
| 이미지 정보가 같은 사진의 JSON끼리 다른 경우, 약 정보가 같은 약의 JSON끼리 다른 경우 | 0 | 5-1의 기준이 새 JSON을 포함한 사본 전체에서도 지켜지는지 확인합니다. |

표 두 개를 더 출력합니다.

- 같은 이미지에서 bbox가 조금이라도 겹치는 쌍(IoU > 0): 원본에서 오류가 아니라고 판단한 2쌍(33880과 35206, 75°와 90° 사진)만 나와야 합니다. 옮기거나 새로 넣은 box가 다른 box와 겹치지 않는지 보려는 것입니다.
- 이미지당 annotation 수의 원본과 사본 비교: 사본은 파일명의 약 코드 수와 같아야 합니다(3개 157장, 4개 75장).

In [ ]:
def load_index(ann_dir):
    """폴더의 JSON을 모두 읽어, JSON 하나가 한 행인 표로 만든다. (3번의 raw_df와 같은 열 + 이미지 크기, JSON 내용)"""
    rows = []
    for path in sorted(ann_dir.rglob("*.json")):
        data = read_json(path)
        img, ann, cat = data["images"][0], data["annotations"][0], data["categories"][0]
        rows.append({
            "json_path": path.relative_to(ann_dir).as_posix(), "folder_code": path.parent.name,
            "file_name": img["file_name"], "image_id": img["id"], "width": img["width"], "height": img["height"],
            "category_id": ann["category_id"], "categories_id": cat["id"], "ann_id": ann["id"],
            "bbox": ann["bbox"], "area": ann["area"], "data": data,
        })
    return pd.DataFrame(rows)


def box_iou(a, b):
    """두 [x, y, w, h] bbox의 IoU (겹친 넓이 ÷ 합친 넓이)"""
    inter_w = max(0, min(a[0] + a[2], b[0] + b[2]) - max(a[0], b[0]))
    inter_h = max(0, min(a[1] + a[3], b[1] + b[3]) - max(a[1], b[1]))
    inter = inter_w * inter_h
    return inter / (a[2] * a[3] + b[2] * b[3] - inter)


proc_df = load_index(OUT_ANN_DIR)
proc_lookup = {(r.file_name, r.category_id): r for r in proc_df.itertuples(index=False)}
n_add = int((fix_df["action"] == "add").sum())

checks = []


def check(name, expected, actual):
    checks.append({"검사": name, "기대값": expected, "결과": actual, "통과": expected == actual})


check("사본 JSON 수", len(raw_df) + n_add, len(proc_df))

not_applied = [r.no for r in fix_df.itertuples(index=False)
               if (r.file_name, r.category_id) not in proc_lookup
               or list(proc_lookup[(r.file_name, r.category_id)].bbox) != r.bbox_after
               or proc_lookup[(r.file_name, r.category_id)].area != r.area_after]
check("수정 목록대로 바뀌지 않은 줄", 0, len(not_applied))

classes_by_image = proc_df.groupby("file_name")["category_id"].apply(set)
code_mismatch = [fn for fn, classes in classes_by_image.items()
                 if classes != {int(code) for code in combo_of(fn).split("-")[1:]}]
check("파일명의 약 코드와 annotation의 클래스가 다른 이미지", 0, len(code_mismatch))

boxes = pd.DataFrame(proc_df["bbox"].tolist(), columns=["x", "y", "w", "h"])
outside = (boxes["x"] < 0) | (boxes["y"] < 0) | (boxes["x"] + boxes["w"] > proc_df["width"]) \
    | (boxes["y"] + boxes["h"] > proc_df["height"])
check("이미지 밖 bbox", 0, int(outside.sum()))
check("area가 너비 × 높이와 다른 annotation", 0, int((proc_df["area"] != boxes["w"] * boxes["h"]).sum()))

overlap_rows = []
for fn, group in proc_df.groupby("file_name"):
    items = list(zip(group["category_id"], group["bbox"]))
    for i in range(len(items)):
        for j in range(i + 1, len(items)):
            value = box_iou(items[i][1], items[j][1])
            if value > 0:
                overlap_rows.append({"file_name": fn, "클래스 1": items[i][0], "bbox 1": items[i][1],
                                     "클래스 2": items[j][0], "bbox 2": items[j][1], "IoU": round(value, 4)})
check("같은 이미지에서 좌표가 같은 서로 다른 클래스의 bbox 쌍", 0, sum(row["IoU"] == 1 for row in overlap_rows))

check("annotation id 중복", 0, int(proc_df["ann_id"].duplicated().sum()))
check("image_id가 2개 이상인 이미지", 0, int((proc_df.groupby("file_name")["image_id"].nunique() > 1).sum()))

proc_folder_id = proc_df["folder_code"].str.removeprefix("K-").astype(int)
check("약 폴더 이름, category_id, categories의 id가 다른 JSON", 0,
      int(((proc_folder_id != proc_df["category_id"]) | (proc_df["categories_id"] != proc_df["category_id"])).sum()))

proc_fields_df = pd.DataFrame([
    {"_image": d["images"][0]["file_name"], "_class": d["annotations"][0]["category_id"],
     **{key: json.dumps(value, ensure_ascii=False) for key, value in d["images"][0].items()}}
    for d in proc_df["data"]
])
check("이미지 정보가 같은 사진의 JSON끼리 다른 경우", 0,
      int(sum((proc_fields_df.groupby("_image")[f].nunique() > 1).sum() for f in IMAGE_FIELDS)))
check("약 정보가 같은 약의 JSON끼리 다른 경우", 0,
      int(sum((proc_fields_df.groupby("_class")[f].nunique() > 1).sum() for f in drug_fields)))

check_df = pd.DataFrame(checks)
with pd.option_context("display.max_colwidth", None):
    display(check_df)
    if overlap_rows:
        display(pd.DataFrame(overlap_rows))

ann_count = pd.DataFrame({
    "원본": raw_df.groupby("file_name").size().value_counts(),
    "사본": proc_df.groupby("file_name").size().value_counts(),
}).fillna(0).astype(int).sort_index()
ann_count.index.name = "이미지당 annotation 수"
display(ann_count)

assert check_df["통과"].all(), f"통과하지 못한 검사: {check_df.loc[~check_df['통과'], '검사'].tolist()}"
print(f"사본 라벨 검사 {len(check_df)}개 모두 통과")

### 6-2. 원본과 복사 검사

원본은 그대로이고, 사본은 고친 곳만 원본과 다른지 확인합니다. 파일 내용 비교는 MD5 해시로 합니다.

| 검사 | 기대값 | 이유 |
|---|---|---|
| 원본 파일 중 바뀌거나, 없어지거나, 새로 생긴 파일 | 0 | 5-2에서 복사 전에 계산한 해시와 비교해, 이 노트북이 원본을 건드리지 않았는지 확인합니다. |
| 사본 `train_images`, `test_images` 중 원본과 다른 파일 | 0 | 이미지는 바꾸지 않고 그대로 복사했습니다. |
| 고치지 않은 JSON 중 원본과 다른 파일 | 0 | 수정 목록에 없는 JSON 750개는 원본과 같아야 합니다. |
| 원본에 없고 사본에만 있는 JSON | 8 | 새로 만든 JSON 8개뿐이어야 합니다. |
| 고친 JSON 중 원본과 같은 파일 | 0 | 고친 JSON 13개가 실제로 바뀌었는지 확인합니다. |
| 고친 JSON 중 bbox, area 말고 다른 값도 바뀐 파일 | 0 | 고칠 값 외에는 건드리지 않았는지 확인합니다. |

원본과 사본의 해시를 다시 계산하므로 시간이 조금 걸립니다.

In [ ]:
def diff_count(before, after):
    """{경로: 해시} 두 개에서 해시가 다르거나 한쪽에만 있는 경로 수"""
    return sum(before.get(key) != after.get(key) for key in set(before) | set(after))


raw_hashes_after = {name: file_hashes(RAW_DIR / name) for name in SUB_DIRS}
proc_hashes = {name: file_hashes(OUT_DIR / name) for name in SUB_DIRS}

raw_json_hashes = raw_hashes_before["train_annotations"]
proc_json_hashes = proc_hashes["train_annotations"]
changed_json = {raw_lookup[(r.file_name, r.category_id)].json_path
                for r in fix_df.itertuples(index=False) if r.action != "add"}
added_json = set(applied_df.loc[applied_df["action"] == "add", "사본 JSON"])
proc_only = set(proc_json_hashes) - set(raw_json_hashes)


def changed_besides_bbox_area(json_rel):
    """고친 JSON에서 bbox와 area를 원본 값으로 되돌렸을 때 원본과 다르면 True"""
    original = read_json(ANN_DIR / json_rel)
    copied = read_json(OUT_ANN_DIR / json_rel)
    for key in ("bbox", "area"):
        copied["annotations"][0][key] = original["annotations"][0][key]
    return copied != original


checks = []
check("원본 파일 중 바뀌거나, 없어지거나, 새로 생긴 파일", 0,
      sum(diff_count(raw_hashes_before[name], raw_hashes_after[name]) for name in SUB_DIRS))
for name in ["train_images", "test_images"]:
    check(f"사본 {name} 중 원본과 다른 파일", 0, diff_count(raw_hashes_before[name], proc_hashes[name]))
check("고치지 않은 JSON 중 원본과 다른 파일", 0,
      sum(raw_json_hashes[key] != proc_json_hashes.get(key) for key in set(raw_json_hashes) - changed_json))
check("원본에 없고 사본에만 있는 JSON", n_add, len(proc_only))
check("사본에만 있는 JSON이 새로 만든 JSON과 같은 목록인지", True, proc_only == added_json)
check("고친 JSON 중 원본과 같은 파일", 0, sum(raw_json_hashes[key] == proc_json_hashes[key] for key in changed_json))
check("고친 JSON 중 bbox, area 말고 다른 값도 바뀐 파일", 0, sum(changed_besides_bbox_area(key) for key in changed_json))

check_df = pd.DataFrame(checks)
with pd.option_context("display.max_colwidth", None):
    display(check_df)
assert check_df["통과"].all(), f"통과하지 못한 검사: {check_df.loc[~check_df['통과'], '검사'].tolist()}"
print(f"원본과 복사 검사 {len(check_df)}개 모두 통과")

## 7. 수정 전후 비교 그림

수정 목록에 있는 이미지 19장마다 원본 라벨(왼쪽)과 사본 라벨(오른쪽)을 나란히 그립니다. 6번은 숫자로 확인한 것이고, 여기서는 고친 box가 실제로 알약을 감싸는지 눈으로 확인합니다. 고친 값의 일부는 raewoo 노트북에서 가져온 것이라, 그림으로 직접 확인하는 단계가 필요합니다.

| 표시 | 뜻 |
|---|---|
| 연두 box | 수정 목록에 없는 annotation (원본과 사본이 같음) |
| 왼쪽 빨간 box | 수정 목록에 있는 annotation의 원본 위치 |
| 오른쪽 파란 box | 고친 위치 (`add`는 새로 넣은 box) |
| box 위 글자 | 약 코드 (K-코드) |

- 그림 안 글자는 영어로 씁니다. EDA 노트북과 같은 기준이며, matplotlib 기본 글꼴에는 한글이 없어서 깨지기 때문입니다.
- 9번의 원본 bbox는 이미지 밖(x = 6567)에 있어서 그릴 수 없으므로, 제목에 적습니다.
- 10–12번 원본에는 좌표가 같은 box가 두 개씩 있습니다. 고친 annotation의 box를 나중에 그려 위에 보이게 하고, 같은 좌표의 box 글자는 위아래로 쌓아서 서로 가리지 않게 합니다.
- 17–19번은 area만 고쳤기 때문에 box 위치가 같습니다. 양쪽 그림이 같아야 하고, 제목에 area 변화를 적습니다.
- 한 그림에 이미지 4장씩, 그림 5개로 나눠 보여 줍니다.

그림에서 확인할 것

- 파란 box가 알약 전체를 감싸고, 다른 알약이나 빈 바닥을 감싸지 않는지
- 파란 box의 여백이 같은 사진의 연두 box와 비슷한지
- 14번에서 K-027733 box가 A1 알약에, K-035206 box가 337 알약에 있는지

In [ ]:
from collections import Counter

import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from PIL import Image

PER_PAGE = 4  # 그림 하나에 넣을 이미지 수


def draw_labels(ax, image, boxes, highlight, color, title):
    """이미지 위에 (category_id, bbox) 목록을 그린다. highlight에 든 클래스는 color로 굵게 그린다."""
    ax.imshow(image)
    stacked = Counter()  # 좌표가 같은 box의 글자를 위로 쌓아서 서로 가리지 않게 합니다.
    # 고친 annotation의 box를 나중에 그려서, 좌표가 같은 다른 box에 가려지지 않게 합니다.
    for class_id, (x, y, w, h) in sorted(boxes, key=lambda item: item[0] in highlight):
        marked = class_id in highlight
        edge = color if marked else "lime"
        ax.add_patch(Rectangle((x, y), w, h, fill=False, edgecolor=edge, linewidth=3 if marked else 1.5))
        ax.text(x, y - 8 - 40 * stacked[(x, y, w, h)], f"K-{class_id:06d}", fontsize=8, color="white",
                va="bottom", clip_on=True,
                bbox=dict(facecolor=edge if marked else "green", alpha=0.85, pad=1.5, linewidth=0))
        stacked[(x, y, w, h)] += 1
    ax.set_xlim(0, image.width)   # 이미지 밖 box는 잘려서 보이지 않습니다.
    ax.set_ylim(image.height, 0)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_title(title, fontsize=9, loc="left")


def change_text(r):
    """수정 목록 한 줄을 그림 제목용 영어 문장으로 만든다."""
    code = f"K-{r.category_id:06d}"
    if r.action == "add":
        return f"{code} added: {r.bbox_after}"
    if r.action == "bbox":
        return f"{code} bbox: {r.bbox_before} -> {r.bbox_after}"
    return f"{code} area only: {r.area_before} -> {r.area_after}"


# 수정 목록의 이미지를 번호 순서로 놓습니다.
image_order = fix_df.groupby("file_name", sort=False)["no"].min().sort_values()

for start in range(0, len(image_order), PER_PAGE):
    page = image_order.iloc[start:start + PER_PAGE]
    fig, axes = plt.subplots(len(page), 2, figsize=(12, 8 * len(page)), layout="constrained", squeeze=False)

    for row_axes, (fn, no) in zip(axes, page.items()):
        rows = fix_df[fix_df["file_name"] == fn]
        highlight = set(rows["category_id"])
        raw_boxes = list(zip(raw_df.loc[raw_df["file_name"] == fn, "category_id"], raw_df.loc[raw_df["file_name"] == fn, "bbox"]))
        proc_boxes = list(zip(proc_df.loc[proc_df["file_name"] == fn, "category_id"], proc_df.loc[proc_df["file_name"] == fn, "bbox"]))

        before = Image.open(IMG_DIR / fn)
        after = Image.open(OUT_DIR / "train_images" / fn)

        # 이미지 밖 bbox는 그릴 수 없으므로 제목에 적습니다.
        outside_notes = [f"K-{c:06d} raw bbox {b} is outside the image" for c, b in raw_boxes
                         if b[0] < 0 or b[1] < 0 or b[0] + b[2] > before.width or b[1] + b[3] > before.height]

        draw_labels(row_axes[0], before, raw_boxes, highlight, "red",
                    "\n".join([f"#{no} BEFORE (data/raw)", fn, *outside_notes]))
        draw_labels(row_axes[1], after, proc_boxes, highlight, "deepskyblue",
                    "\n".join([f"#{no} AFTER (data/processed)", *[change_text(r) for r in rows.itertuples(index=False)]]))

    fig.suptitle(f"Label fixes: images {start + 1}-{start + len(page)} of {len(image_order)} "
                 "(red = fixed annotation before, blue = after, green = unchanged)", fontsize=11)
    plt.show()

## 8. 클래스별 bbox 영역 모아 보기 (사본)

EDA 노트북 5-5의 클래스별 bbox 영역 그림을, 고친 사본으로 다시 만듭니다. 확인하려는 것은 두 가지입니다.

- EDA 5-5에서 다르게 보였던 6칸이, 사본에서는 같은 클래스의 다른 칸과 같은 알약으로 보이는지
    - 3351 #97(바닥만 보임), 3351 #129(흰색 알약), 16548 #3(S300), 20238 #7(HT), 27733 #10(337), 35206 #29(A1)
- 새로 넣은 box 8개와 이미지 밖에서 고친 box(18357)가 각 클래스의 다른 칸과 같은 알약인지

그리는 방식은 EDA 5-5와 같습니다.

- bbox 영역을 잘라 100px 칸에 맞게 줄이고, 한 줄에 12칸씩 클래스별로 이어 붙입니다.
- 칸 왼쪽 위의 빨간 번호는 그 클래스 안에서의 순서입니다. 순서는 JSON 파일 경로 순서입니다.
- 그림 하나에 클래스 8개씩, 그림 7개로 나눠 저장하고, 노트북에는 파일 이름과 클래스 목록만 출력합니다.

EDA 5-5와 다른 점은 다음과 같습니다.

- 사본에는 이미지 밖 bbox가 없어서 annotation 771개를 모두 자릅니다. EDA에서는 18357의 1개를 빼서 762개였습니다.
- 새로 넣은 box 때문에 일부 클래스(3351, 4543, 20014, 32310, 33880)와, 이번에 들어간 18357은 칸 번호가 EDA 그림과 달라집니다. 그래서 고친 annotation이 그림에서 몇 번 칸인지 표로 출력합니다.
- 저장 위치는 `notebooks/eda/label_fix/fix_class_bbox_crops/`이고, 폴더가 없으면 만듭니다. 파일 이름은 `fix_class_bbox_crops_01.png` ~ `fix_class_bbox_crops_07.png`입니다.
- 다시 실행하면 그림을 덮어씁니다. EDA 5-5는 그림이 이미 있으면 건너뛰었지만, 이 노트북은 실행할 때마다 사본을 다시 만들기 때문에 그림도 지금의 사본과 같게 맞춥니다.

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw

CELL = 100       # 잘라 낸 영역 하나를 넣을 칸의 크기(픽셀)
NCOLS = 12       # 한 줄에 넣을 칸 수
CLASS_PAGE = 8   # 그림 하나에 넣을 클래스 수
CROP_DIR = FIX_CSV.parent / "fix_class_bbox_crops"   # notebooks/eda/label_fix/fix_class_bbox_crops
CROP_DIR.mkdir(exist_ok=True)

# 사본 annotation 771개 (6-1의 proc_df, JSON 파일 경로 순서). 칸 번호는 클래스 안에서의 순서입니다.
crop_df = proc_df[["json_path", "file_name", "category_id", "bbox"]].reset_index(drop=True)
crop_df["칸 번호"] = crop_df.groupby("category_id").cumcount()

# 모든 bbox 영역을 잘라 칸 크기에 맞게 줄여 둡니다. 이미지는 한 장씩 한 번만 엽니다.
crops = {}
for file_name, rows in crop_df.groupby("file_name"):
    with Image.open(OUT_DIR / "train_images" / file_name) as img:
        img = img.convert("RGB")
        for i, r in rows.iterrows():
            x, y, w, h = r["bbox"]
            crop = img.crop((x, y, x + w, y + h))  # (왼쪽, 위, 오른쪽, 아래) 영역을 잘라 냄
            crop.thumbnail((CELL - 4, CELL - 4))   # 비율을 유지한 채 칸 안에 들어가게 줄임
            crops[i] = crop


def class_montage(class_id):
    """한 클래스의 잘라 낸 영역을 한 줄에 NCOLS개씩 붙인 이미지를 만든다. 칸마다 순서 번호를 적는다."""
    idx = crop_df.index[crop_df["category_id"] == class_id]
    nrows = (len(idx) + NCOLS - 1) // NCOLS
    canvas = Image.new("RGB", (NCOLS * CELL, nrows * CELL), "white")
    draw = ImageDraw.Draw(canvas)
    for k, i in enumerate(idx):
        c = crops[i]
        col, row = k % NCOLS, k // NCOLS
        canvas.paste(c, (col * CELL + (CELL - c.width) // 2, row * CELL + (CELL - c.height) // 2))
        draw.text((col * CELL + 3, row * CELL + 2), str(k), fill="red")
    return canvas


class_ids = sorted(crop_df["category_id"].unique())
class_names = {d["annotations"][0]["category_id"]: d["images"][0]["dl_name"] for d in proc_df["data"]}
proc_count = crop_df["category_id"].value_counts()
raw_count = raw_df["category_id"].value_counts()
page_of = {}  # 클래스 → 저장한 그림 파일 이름

for page_no, start in enumerate(range(0, len(class_ids), CLASS_PAGE), 1):
    page_ids = class_ids[start:start + CLASS_PAGE]
    out_path = CROP_DIR / f"fix_class_bbox_crops_{page_no:02d}.png"
    montages = [class_montage(cid) for cid in page_ids]
    heights = [m.height for m in montages]

    # 클래스마다 줄 수가 달라서, 각 묶음의 높이를 줄 수에 맞춰 나눕니다.
    fig, axes = plt.subplots(len(page_ids), 1, figsize=(12, sum(heights) / 100 + 0.4 * len(page_ids)),
                             gridspec_kw={"height_ratios": heights}, layout="constrained", squeeze=False)
    for ax, cid, m in zip(axes[:, 0], page_ids, montages):
        ax.imshow(m)
        ax.set_title(f"category_id {cid} (n={proc_count[cid]})", loc="left", fontsize=9)
        ax.axis("off")
    fig.savefig(out_path, dpi=100)
    plt.close(fig)  # 파일로만 저장하고, 노트북에는 그림을 표시하지 않습니다.

    print(f"{rel(out_path)}  (클래스 {start + 1}~{start + len(page_ids)})")
    for cid in page_ids:
        page_of[cid] = out_path.name
        print(f"    {cid}: {class_names[cid]} ({proc_count[cid]}개, 원본 {raw_count[cid]}개)")

# 고친 annotation이 그림에서 몇 번 칸인지 표로 보여 줍니다.
fixed_cells = crop_df.merge(fix_df[["no", "file_name", "category_id", "action"]], on=["file_name", "category_id"])
fixed_cells["그림 파일"] = fixed_cells["category_id"].map(page_of)
with pd.option_context("display.max_colwidth", None):
    display(fixed_cells.sort_values("no")[["no", "action", "category_id", "칸 번호", "그림 파일", "file_name"]]
            .reset_index(drop=True))